# CivicDesk 311 Ticket Rewriter — LoRA SFT + DPO on a free-tier Colab GPU

Adapts Qwen2.5-0.5B-Instruct with LoRA to rewrite messy resident 311 complaints into structured dispatch tickets, then runs a short DPO pass on top of the same adapter.

**Before running:** Runtime > Change runtime type > T4 GPU (free tier).

Every training run below writes `run_metrics.json` (peak GPU memory + wall-clock time) next to its checkpoint, and the evaluation step records generation wall-clock time — this is the evidence that the whole pipeline fits the free-tier hardware budget.

In [ ]:
!nvidia-smi

## 1. Clone the repo

In [ ]:
!git clone https://github.com/nmilleson/csp-cst627-week3.git
%cd csp-cst627-week3

## 2. Install dependencies

Colab GPU runtimes ship a CUDA-matched `torch` already, so this just adds the rest on top.

In [ ]:
!pip install -q -r requirements.txt

Colab's base image preinstalls an old `torchao` (0.10.x). Newer `peft` raises an `ImportError` (instead of skipping gracefully) when it hits that incompatible version while dispatching LoRA onto a plain, non-quantized `nn.Linear` layer — which is exactly what happens in `evaluate.py` when it loads a model without 4-bit quantization. Upgrading it here avoids that crash.

In [ ]:
!pip install -q -U "torchao>=0.16.0"

In [ ]:
import torch
print("torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("total memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

## 3. Dataset

`data/train.jsonl`, `data/val.jsonl`, `data/dpo_train.jsonl`, and `data/dpo_val.jsonl` are already committed to the repo. This cell regenerates them from scratch with the same seeds, just to show the pipeline is fully reproducible end to end.

In [ ]:
!python data/generate_dataset.py --n-per-category 30 --seed 7 --output-dir data
!python data/generate_preferences.py --input data/train.jsonl --output data/dpo_train.jsonl --seed 11
!python data/generate_preferences.py --input data/val.jsonl --output data/dpo_val.jsonl --seed 12

## 4. LoRA SFT pass

QLoRA (4-bit) fine-tune of Qwen2.5-0.5B-Instruct on the 270 training pairs. LoRA rank 16 keeps the trainable adapter to a few million parameters.

In [ ]:
!python training/sft_train.py \
    --train data/train.jsonl --val data/val.jsonl \
    --output-dir checkpoints/sft-adapter

In [ ]:
import json
print(json.dumps(json.load(open("checkpoints/sft-adapter/run_metrics.json")), indent=2))

## 5. DPO pass on top of the SFT adapter

Continues training the *same* adapter with preference pairs (chosen vs. rejected tickets), using the frozen base model (adapter disabled) as the implicit reference policy — no second full model copy needed.

In [ ]:
!python training/dpo_train.py \
    --sft-adapter checkpoints/sft-adapter \
    --train data/dpo_train.jsonl --val data/dpo_val.jsonl \
    --output-dir checkpoints/dpo-adapter

In [ ]:
print(json.dumps(json.load(open("checkpoints/dpo-adapter/run_metrics.json")), indent=2))

## 6. Evaluate base vs. SFT vs. DPO

Same 30 held-out complaints, greedy decoding, scored for JSON validity, schema completeness, and category/priority accuracy against the ground-truth ticket.

In [ ]:
!python training/evaluate.py --label base --eval-file data/val.jsonl --output eval_results/base
!python training/evaluate.py --label sft --adapter checkpoints/sft-adapter --eval-file data/val.jsonl --output eval_results/sft
!python training/evaluate.py --label dpo --adapter checkpoints/dpo-adapter --eval-file data/val.jsonl --output eval_results/dpo

In [ ]:
!python training/compare_results.py eval_results/base.summary.json eval_results/sft.summary.json eval_results/dpo.summary.json

## 7. Hardware budget summary

Pull the peak-memory/wall-clock numbers from every run into one place for the write-up to Elena and the platform team.

In [ ]:
sft_metrics = json.load(open("checkpoints/sft-adapter/run_metrics.json"))
dpo_metrics = json.load(open("checkpoints/dpo-adapter/run_metrics.json"))

print(f"SFT   — {sft_metrics['wall_clock_seconds']:.1f}s, peak GPU memory {sft_metrics['peak_gpu_memory_gb']:.2f} GB")
print(f"DPO   — {dpo_metrics['wall_clock_seconds']:.1f}s, peak GPU memory {dpo_metrics['peak_gpu_memory_gb']:.2f} GB")
print("Free-tier Colab T4 has 16 GB VRAM.")

## 8. Save artifacts

Colab runtimes are ephemeral — zip the adapters and eval results so they survive the session (download, or copy to Drive).

In [ ]:
!zip -qr artifacts.zip checkpoints/sft-adapter checkpoints/dpo-adapter eval_results
from google.colab import files
files.download("artifacts.zip")